# Notebook to serve the best model

In [62]:
from pathlib import Path
import sys
import joblib
from skimage import io
from skimage.transform import resize
import gc

## Setup project root and load module

In [11]:
%%capture --no-display
%load_ext autoreload
%autoreload 2

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from weedcrop.autofarm import *

In [15]:
project_root

PosixPath('/home/master/dev/demokritos/A/ml_course/ml_assignment_autofarm')

## Load models

In [24]:
scaler = joblib.load(project_root / "models/scaler.joblib")
pca = joblib.load(project_root / "models/pca.joblib")
model = joblib.load(project_root / "models/svc_opt.joblib")

## Load random test image

In [19]:
img_path = project_root / "data/image.jpg"

In [44]:
import requests
img_url = "https://media.sciencephoto.com/image/c0124709/800wm/C0124709-Weeds_in_sugar_beet_crop.jpg"
resp = requests.get(img_url)
resp.raise_for_status()

with open(img_path, "wb") as f:
    f.write(resp.content)
    
# Load to memory
img = io.imread(img_path)
    
# Resize to target_size=(256, 256)
img = resize(img, (256, 256), anti_aliasing=True)

# Convert for compatibility
img_lst = [img]

## Extract features

In [66]:
# Extract features
img_hog = extract_hog_features_from_list(img_lst)
img_lbp = extract_lbp_features_from_list(img_lst)
img_hsv = extract_hsv_features_from_list(img_lst)

In [67]:
# Apply PCA on HOG: pre-fitted scaler and pca
img_hog_reduced = reduce_hog_features(
    img_hog, 
    fitted_scaler=scaler, 
    fitted_pca=pca
)

In [68]:
# Combine features
img_feat_combined = feature_fusion(feature_list=[img_lbp, img_hog_reduced, img_hsv])

In [69]:
# Cleanup
list(map(len, (img_feat_combined[0], img_hog_reduced[0], img_lbp[0], img_hsv[0])))
del img_hog, img_hog_reduced, img_lbp, img_hsv
gc.collect()

2948

## Test prediction

In [98]:
opt_threshold = 0.023699424100647747 # Load optimized threshold from experiments
probs = model.predict_proba(img_feat_combined)
crop_prob = probs[0, 0]

if crop_prob >= opt_threshold:
    prediction = "crop"
    confidence = crop_prob
else:
    prediction = "weed"
    confidence = probs[0, 1]
    
print(prediction)
print(f"Confidence: {confidence:.4%}")

weed
Confidence: 99.9999%


---

## Put on Gardio

In [100]:
import gradio as gr
import numpy as np
import joblib
from PIL import Image
from skimage.transform import resize
from huggingface_hub import hf_hub_download

from weedcrop.autofarm import (
    extract_hog_features_from_list,
    extract_lbp_features_from_list,
    extract_hsv_features_from_list,
    feature_fusion
)

In [102]:
# Load models
REPO_ID = "ipetrousov/weedcrop_svm_classifier"

model_file = hf_hub_download(repo_id=REPO_ID, filename="svc_opt.joblib")
scaler_file = hf_hub_download(repo_id=REPO_ID, filename="scaler.joblib")
pca_file = hf_hub_download(repo_id=REPO_ID, filename="pca.joblib")

model = joblib.load(model_file)
scaler = joblib.load(scaler_file)
pca = joblib.load(pca_file)

In [103]:
OPT_THRESHOLD = 0.023699424100647747

def classify_plant(image_input):
    if image_input is None:
        return "No image provided", "0.00%"

    # Convert to NumPy array and resize to target dimension (256, 256)
    img_np = np.array(image_input)
    img_resized = resize(img_np, (256, 256), anti_aliasing=True)
    img_list = [img_resized]

    # Feature extraction pipeline
    img_hog = extract_hog_features_from_list(img_list)
    img_lbp = extract_lbp_features_from_list(img_list)
    img_hsv = extract_hsv_features_from_list(img_list)

    # Dimensionality reduction on HOG features (Testing mode using fitted scaler & pca)
    img_hog_scaled = scaler.transform(img_hog)
    img_hog_reduced = pca.transform(img_hog_scaled)

    # Multi-modal feature fusion
    X_combined = feature_fusion(
        feature_list=[img_lbp, img_hog_reduced, img_hsv]
    )

    # Probabilistic inference
    probs = model.predict_proba(X_combined)
    crop_prob = probs[0, 0]

    # Apply surgical threshold logic
    if crop_prob >= OPT_THRESHOLD:
        prediction = "CROP 🌿"
        confidence = crop_prob
    else:
        prediction = "WEED 🌾"
        confidence = probs[0, 1]

    confidence_str = f"{confidence:.4%}"
    detailed_metrics = (
        f"Decision: {prediction}\n"
        f"Confidence: {confidence_str}\n"
        f"Raw Crop Probability: {crop_prob:.6f}\n"
        f"Operating Threshold: {OPT_THRESHOLD:.6f}"
    )

    return prediction, detailed_metrics

# 3. Gradio Interface Construction
demo = gr.Interface(
    fn=classify_plant,
    inputs=gr.Image(type="pil", label="Upload Seedling Image"),
    outputs=[
        gr.Textbox(label="Predicted Class"),
        gr.Textbox(label="Evaluation Breakdown")
    ],
    title="Autofarm: Surgical Weed & Crop Classifier",
    description="Drop a plant image to evaluate it using an optimized SVM classifier with multi-modal feature fusion (HOG + LBP + HSV).",
    examples=[]
)

In [104]:
# Run model
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
